# IV-4 確率と統計的信号処理（ノートブック版）

教科書の IV-4 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。全部で 1〜2 分です）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 複素正規分布（密度の照合、円対称、パワーの分布、IS ダイバージェンスとの関係、STFT の端のビン） | B | `cgauss.py` |
| 2 | 自己相関とパワースペクトル密度（ウィーナー–ヒンチン、ピリオドグラムとウェルチ法） | C | `psd.py` |
| 3 | 混合正規分布の EM と sklearn の照合 | D | `em_gmm.py` |
| 4 | 補助関数法（MM アルゴリズム）の3つの例 | D | `mm_demo.py` |
| 5 | 音声の統計を見て、聞く | E | `speech_stats.py` |
| 6 | ウィーナーフィルタと時間周波数マスク | F | `wiener.py` |
| 7 | 確認問題 10 のコード | 7 節 | （解答の中のコード） |

- 次のセルたちが、IV-2 の共通ライブラリ（`common/` の `__init__.py`・`stft.py`・`audio_io.py`）を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の IV-2 の 6 節と同じなので、ほかのファイルは要りません（IV-4 では `plotting.py` は使わないので書き出しません）。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックの隣を使う）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、日本語のフォントの候補を足すこと、コマンドの引数を `sys.argv` で渡すこと、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」を確かめることだけです。手順 B の 3（わざと間違えてみる）と C の 6（区間の長さを変える）は、元のプログラムを書きかえる代わりに、小さなセルを足して同じことを確かめます。
- **入れていないもの**：手順 E の 11（第7章で録った自分の声で `speech_stats.py` をくり返す）と、自分の録音での `wiener.py` は、録音のファイルがないので入れていません。Colab なら、左の「ファイル」に wav を上げて、`sys.argv` に `"my_voice.wav"` を足せば試せます（画面の項目名は要確認）。Mac のターミナルでは教科書の手順どおりに実行してください。手順 G（論文を読む）にはコードはありません。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。
- 話し声（5・6 節）は librosa の例の音源 `libri1` を使います（初回はダウンロードします。ネットワークが要ります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、scipy・scikit-learn・librosa の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・scikit-learn・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, sklearn, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "sklearn", sklearn.__version__, "librosa", librosa.__version__)
print("（教科書は scipy 1.18.1、scikit-learn 1.9.1、librosa 1.0.1 で確かめた）")
import os
os.makedirs("common", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common.stft import stft, istft
from common.audio_io import save, load
print("共通ライブラリを読み込めました（stft・istft・save・load）")


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.31, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが同じ名前を使っても、確かめのセルで戻す

## 1. 複素正規分布（IV-4-3-3・3-7、手順 B、`cgauss.py`）

自前の複素正規分布の対数密度を、実部・虚部に分けた 6 次元の正規分布（scipy）と照合し、乱数・パワーの分布・IS ダイバージェンスとの関係・STFT の端のビンを確かめます（数秒）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-4：複素正規分布を NumPy で自分で実装し、scipy（実部・虚部に分けた 2M 次元の正規分布）と照合する

使い方: cd ~/home-lab/partIV/iv4 && uv run python cgauss.py
やること
  1. 1変数の数値例（σ² = 2、z = 1 + j）の密度
  2. M = 3 の複素正規分布の対数密度を、scipy.stats.multivariate_normal（2M 次元の実の正規分布）と照合
  3. 乱数を作り、共分散 Σ・擬似共分散 C（円対称なら 0）・実部の分散（Σ の対角の半分）を確かめる
  4. パワー |z|² が平均 σ² の指数分布、位相が一様分布になることを確かめる
  5. 分散の最尤推定 v̂ = 平均 |z|² の分散が、クラメール・ラオの下限 v²/T に一致することを確かめる
  6. 負の対数尤度と IS ダイバージェンス、ポアソン分布と KL ダイバージェンスの「差が定数」を確かめる
  7. 実数の信号の STFT で、直流とナイキストのビンは円対称でないことを確かめる（共通ライブラリ stft.py を使う）
"""
import pathlib
import sys

import numpy as np
from scipy import stats
from scipy.special import gammaln

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft                                             # IV-2 の共通ライブラリ

RNG = np.random.default_rng(0)


def crandn(n, Sigma, rng=RNG):
    """平均 0、共分散 Sigma（M×M、エルミートで正定値）の円対称な複素正規分布の乱数を n 個。形は (n, M)"""
    M = Sigma.shape[0]
    Lc = np.linalg.cholesky(Sigma)                       # Sigma = Lc Lcᴴ
    g = (rng.standard_normal((n, M)) + 1j * rng.standard_normal((n, M))) / np.sqrt(2)   # N_C(0, I)
    return g @ Lc.T                                      # 各行 z = Lc g（行ベクトルなので転置を掛ける）


def logpdf_cn(z, mu, Sigma):
    """複素正規分布 N_C(mu, Sigma) の対数密度。z は (n, M)
    log p = −M log π − log det Σ − (z − μ)ᴴ Σ⁻¹ (z − μ)"""
    M = Sigma.shape[0]
    d = z - mu
    sign, logdet = np.linalg.slogdet(Sigma)               # det Σ は正の実数（sign は 1）
    q = np.real(np.einsum("nm,nm->n", d.conj(), np.linalg.solve(Sigma, d.T).T))
    return -M * np.log(np.pi) - logdet - q


def real_cov(Sigma):
    """円対称な複素正規分布 N_C(0, Σ) を、[Re z; Im z] の 2M 次元の正規分布で表したときの共分散
    Γ = (1/2) [[Re Σ, −Im Σ], [Im Σ, Re Σ]]"""
    return 0.5 * np.block([[Sigma.real, -Sigma.imag], [Sigma.imag, Sigma.real]])


def is_div(p, v):
    """IS ダイバージェンス D_IS(p|v) = p/v − log(p/v) − 1（要素ごと）"""
    r = p / v
    return r - np.log(r) - 1


def kl_div(x, v):
    """一般化 KL ダイバージェンス D_KL(x|v) = x log(x/v) − x + v（x = 0 の項は v）"""
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(x > 0, x * np.log(x / v), 0.0)
    return t - x + v


def main():
    print("[1] 1変数の数値例：N_C(0, σ² = 2) の z = 1 + j での密度")
    p_own = np.exp(logpdf_cn(np.array([[1 + 1j]]), np.zeros(1), np.array([[2.0]])))[0]
    p_real = stats.norm(0, 1).pdf(1) ** 2                 # 実部・虚部はそれぞれ分散 σ²/2 = 1 で独立
    print(f"  自前 {p_own:.6f}、実部×虚部の正規分布 {p_real:.6f}、式 e⁻¹/(2π) = {np.exp(-1) / (2 * np.pi):.6f}")

    print("\n[2] M = 3 の対数密度を scipy（2M = 6 次元の実の正規分布）と照合")
    M = 3
    A = RNG.standard_normal((M, M)) + 1j * RNG.standard_normal((M, M))
    Sigma = A @ A.conj().T + 0.5 * np.eye(M)               # エルミートで正定値
    mu = RNG.standard_normal(M) + 1j * RNG.standard_normal(M)
    z = mu + crandn(1000, Sigma)
    lp_own = logpdf_cn(z, mu, Sigma)
    mvn = stats.multivariate_normal(mean=np.r_[mu.real, mu.imag], cov=real_cov(Sigma))
    lp_sp = mvn.logpdf(np.c_[z.real, z.imag])
    print(f"  Σ はエルミート：max|Σ − Σᴴ| = {np.max(np.abs(Sigma - Sigma.conj().T)):.1e}、固有値 {np.round(np.linalg.eigvalsh(Sigma), 3)}")
    print(f"  max|自前 − scipy| = {np.max(np.abs(lp_own - lp_sp)):.2e}（許容 1e-10）")
    assert np.max(np.abs(lp_own - lp_sp)) < 1e-10

    print("\n[3] 乱数の共分散・擬似共分散（n = 200000）")
    n = 200_000
    z = crandn(n, Sigma)
    S_hat = z.T @ z.conj() / n                             # E[z zᴴ]
    C_hat = z.T @ z / n                                    # E[z zᵀ]（擬似共分散。円対称なら 0）
    print(f"  max|Σ̂ − Σ| = {np.max(np.abs(S_hat - Sigma)):.3f}（Σ の大きさは {np.max(np.abs(Sigma)):.2f}）")
    print(f"  max|擬似共分散| = {np.max(np.abs(C_hat)):.3f}（0 に近い＝円対称）")
    print(f"  実部の分散 {np.var(z[:, 0].real):.3f}、虚部の分散 {np.var(z[:, 0].imag):.3f}、"
          f"Σ₁₁/2 = {Sigma[0, 0].real / 2:.3f}（Σ₁₁ ではない）")

    print("\n[4] 1変数 N_C(0, σ² = 2)：パワーは指数分布、位相は一様分布")
    s2 = 2.0
    z1 = crandn(n, np.array([[s2]]))[:, 0]
    pw = np.abs(z1) ** 2
    ks_p = stats.kstest(pw, stats.expon(scale=s2).cdf).pvalue
    ks_ph = stats.kstest(np.angle(z1), stats.uniform(-np.pi, 2 * np.pi).cdf).pvalue
    print(f"  |z|² の平均 {pw.mean():.3f}（σ² = {s2}）、分散 {pw.var():.3f}（σ⁴ = {s2**2}）、"
          f"E|z|⁴/(E|z|²)² = {np.mean(pw**2) / pw.mean()**2:.3f}（2 になる）")
    print(f"  コルモゴロフ–スミルノフ検定の p 値：パワー vs 指数分布 {ks_p:.2f}、位相 vs 一様分布 {ks_ph:.2f}"
          "（大きい＝矛盾しない。正しさの証明ではない）")

    print("\n[5] 分散の最尤推定 v̂ = (1/T)Σ|z_t|² とクラメール・ラオの下限 v²/T（v = 2、T = 20、10000 回）")
    T, R = 20, 10_000
    zz = crandn(T * R, np.array([[s2]]))[:, 0].reshape(R, T)
    vhat = np.mean(np.abs(zz) ** 2, axis=1)
    print(f"  v̂ の平均 {vhat.mean():.4f}（偏りなし）、分散 {vhat.var():.4f}、下限 v²/T = {s2**2 / T:.4f}")

    print("\n[6] 負の対数尤度とダイバージェンスの差は、パラメータによらない定数")
    F, Tt, K = 20, 30, 3
    W0 = RNG.uniform(0.1, 1, (F, K))
    H0 = RNG.uniform(0.1, 1, (K, Tt))
    V0 = W0 @ H0
    x = np.sqrt(V0 / 2) * (RNG.standard_normal((F, Tt)) + 1j * RNG.standard_normal((F, Tt)))   # x_ft ~ N_C(0, v_ft)
    p = np.abs(x) ** 2
    const_is = np.sum(np.log(np.pi) + np.log(p) + 1)
    for name, V in (("真の V", V0), ("別の V", RNG.uniform(0.1, 2, (F, Tt))), ("さらに別の V", V0 * 3.7)):
        nll = np.sum(np.log(np.pi) + np.log(V) + p / V)
        print(f"  {name:8s}：負の対数尤度 {nll:10.4f}、Σ D_IS {np.sum(is_div(p, V)):10.4f}、"
              f"差 {nll - np.sum(is_div(p, V)):10.6f}（定数 Σ(log π + log p + 1) = {const_is:.6f}）")
    lam = RNG.uniform(0.5, 20, 500)
    cnt = RNG.poisson(lam).astype(float)
    const_kl = np.sum(gammaln(cnt + 1) - np.where(cnt > 0, cnt * np.log(np.maximum(cnt, 1)), 0) + cnt)
    for name, V in (("真の平均", lam), ("別の平均", RNG.uniform(0.5, 20, 500))):
        nll = -np.sum(stats.poisson.logpmf(cnt, V))
        print(f"  ポアソン {name}：負の対数尤度 − Σ D_KL = {nll - np.sum(kl_div(cnt, V)):.6f}"
              f"（定数 Σ(log x! − x log x + x) = {const_kl:.6f}）")
    print(f"  スケール：D_IS(2|1) = {is_div(2.0, 1.0):.4f}、D_IS(200|100) = {is_div(200.0, 100.0):.4f}；"
          f"D_KL(2|1) = {kl_div(np.array(2.0), 1.0):.4f}、D_KL(200|100) = {kl_div(np.array(200.0), 100.0):.4f}")

    print("\n[7] 実数の白色雑音（60 秒分）の STFT（L = 512、S = 128）：ビンごとの円対称性 |E[x²]|/E[|x|²]")
    X = stft(RNG.standard_normal(16000 * 60), L=512, S=128)
    F = X.shape[0]
    for f in (0, 1, 2, 64, F - 2, F - 1):
        c = np.abs(np.mean(X[f] ** 2)) / np.mean(np.abs(X[f]) ** 2)
        print(f"  f = {f:3d}：{c:.3f}" + ("（直流・ナイキストは実数なので 1）" if f in (0, F - 1) else
                                     "（ハン窓では理論値 1/6 ≈ 0.167）" if f in (1, F - 2) else
                                     "（理論値 0。0 でないのは推定のばらつき）"))


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(len(re.findall(r"0\.058550", cap.stdout)) == 3, "[1] 3つの値がどれも 0.058550")
ok(num(r"max\|自前 − scipy\| = ([\d.e+-]+)") < 1e-10, "[2] max|自前 − scipy| が 1e-10 未満（教科書 2.5e-14）")
ok(num(r"max\|擬似共分散\| = ([\d.]+)") < 0.05, "[3] 擬似共分散が 0 に近い（教科書 0.02 程度）")
ok(abs(num(r"実部の分散 ([\d.]+)") - num(r"Σ₁₁/2 = ([\d.]+)")) < 0.05, "[3] 実部の分散が Σ₁₁/2 に近い")
ok(abs(num(r"\|z\|² の平均 ([\d.]+)") - 2) < 0.05 and abs(num(r"）、分散 ([\d.]+)（σ⁴") - 4) < 0.15
   and abs(num(r"E\|z\|⁴/\(E\|z\|²\)² = ([\d.]+)") - 2) < 0.03, "[4] パワーの平均 ≈ 2、分散 ≈ 4、比 ≈ 2.00")
d_is = re.findall(r"差\s+([\d.]+)（定数", cap.stdout)
d_kl = re.findall(r"Σ D_KL = ([\d.]+)", cap.stdout)
ok(len(d_is) == 3 and len(set(d_is)) == 1 and len(d_kl) == 2 and len(set(d_kl)) == 1, f"[6] 差が3行とも同じ定数 {d_is[0]}（IS）、2行とも同じ定数 {d_kl[0]}（ポアソン）")
c7 = [num(rf"f = \s*{f}：([\d.]+)") for f in (0, 1, 2, 64, 255, 256)]
ok(c7[0] == 1 and c7[5] == 1 and 0.14 < c7[1] < 0.19 and 0.14 < c7[4] < 0.19 and c7[2] < 0.05 and c7[3] < 0.05,
   f"[7] {c7}（1.000・約 0.17・0.01 程度・0.01 程度・約 0.16・1.000）")

### 手順 B の 3：わざと間違えてみる（ノートブック版で足したセル）

教科書では `crandn` の `/ np.sqrt(2)` を消して実行します。ここでは、消した版 `crandn_wrong` を別に作って同じことを確かめます（上の `crandn`・`logpdf_cn`・`real_cov` を使います）。

In [ ]:
# ノートブック版で足したセル：crandn の / np.sqrt(2) を消した版
from scipy import stats


def crandn_wrong(n, Sigma, rng):
    M = Sigma.shape[0]
    Lc = np.linalg.cholesky(Sigma)
    g = rng.standard_normal((n, M)) + 1j * rng.standard_normal((n, M))     # / np.sqrt(2) を消した
    return g @ Lc.T


rng = np.random.default_rng(0)
M = 3
A = rng.standard_normal((M, M)) + 1j * rng.standard_normal((M, M))
Sigma = A @ A.conj().T + 0.5 * np.eye(M)
z = crandn_wrong(200_000, Sigma, rng)
ratio = np.real(np.trace(z.T @ z.conj() / len(z))) / np.real(np.trace(Sigma))
pw = np.abs(crandn_wrong(200_000, np.array([[2.0]]), rng)[:, 0]) ** 2
zz = crandn_wrong(1000, Sigma, rng)
lp_gap = np.max(np.abs(logpdf_cn(zz, np.zeros(M), Sigma)
                       - stats.multivariate_normal(mean=np.zeros(2 * M), cov=real_cov(Sigma)).logpdf(np.c_[zz.real, zz.imag])))
print(f"[3] Σ̂ の対角の和 / Σ の対角の和 = {ratio:.3f}（約 2 倍）")
print(f"[4] パワーの平均 {pw.mean():.3f}（σ² = 2 の約 2 倍）")
print(f"[2] 密度の照合 max|自前 − scipy| = {lp_gap:.1e}（通ってしまう）")
ok(abs(ratio - 2) < 0.1, "手順 B の 3：Σ̂ が Σ の約 2 倍")
ok(abs(pw.mean() - 4) < 0.1, "手順 B の 3：パワーの平均が約 4（σ² の 2 倍）")
ok(lp_gap < 1e-10, "手順 B の 3：[2] の照合は通ってしまう（密度の式の照合では、乱数の作り方の誤りは見つからない）")

## 2. 自己相関とパワースペクトル密度（IV-4-3-1・3-2、手順 C、`psd.py`）

十数秒かかります。図の左で、灰色のピリオドグラムが上下に 20〜30 dB も激しくばらつき、ウェルチ法（青）が理論の破線にほぼ重なること、右で、ピリオドグラム／真の値のヒストグラムが指数分布 $e^{-q}$ に重なり、ウェルチ法は 1 のまわりに細く集まることを見てください（手順 C の 5）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-4：自己相関とパワースペクトル密度。ピリオドグラムとウェルチ法を自分で実装し、scipy.signal.welch と照合する

使い方: cd ~/home-lab/partIV/iv4 && uv run python psd.py
やること
  1. 自前のウェルチ法と scipy.signal.welch の照合（片側、V²/Hz、区間ごとに平均を引く）
  2. ウィーナー–ヒンチンの定理の有限版：ピリオドグラム ＝ 標本自己相関のフーリエ変換（ちょうど等しい）
  3. AR(1) 過程の理論の PSD と、ピリオドグラム・ウェルチ法の推定を比べる（psd.png）
  4. ピリオドグラムの分散は長く録っても減らない／ウェルチ法の分散と自由度、95% 区間
  5. 誤用1：平均しないピリオドグラムの最大のピーク（白色雑音でも約 +10 dB）
  6. 誤用2：自己相関のある時系列の平均の分散を、独立とみなして計算する
  7. 誤解：dB にしてから平均すると、約 −2.5 dB 小さく出る
"""
import numpy as np
from scipy import signal, stats
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(1)
FS = 16000
EULER = 0.5772156649


def periodogram(x, fs=1.0, window=None):
    """片側のピリオドグラム（V²/Hz）。window が None なら矩形窓"""
    N = len(x)
    w = np.ones(N) if window is None else window
    X = np.fft.rfft(w * x)
    P = np.abs(X) ** 2 / (fs * np.sum(w ** 2))
    P[1:-1 if N % 2 == 0 else None] *= 2          # 負の周波数の分を足す（直流と、偶数長ならナイキストは足さない）
    return np.fft.rfftfreq(N, 1 / fs), P


def welch(x, fs=1.0, L=256, window="hann"):
    """自前のウェルチ法：長さ L、重なり L/2 の区間に分け、平均を引き、窓を掛けたピリオドグラムを平均する"""
    w = signal.get_window(window, L)              # 周期的なハン窓（scipy の既定と同じ）
    S = L // 2
    K = (len(x) - L) // S + 1                     # 区間の数（端の余りは使わない）
    P = 0.0
    for k in range(K):
        seg = x[k * S:k * S + L]
        f, Pk = periodogram(seg - seg.mean(), fs, w)
        P = P + Pk
    return f, P / K, K


def ar1(n, a, rng=RNG):
    """x[n] = a x[n−1] + e[n]、e は分散 1 の白色雑音。定常な状態から始める"""
    e = rng.standard_normal(n)
    e[0] *= 1 / np.sqrt(1 - a ** 2)
    return signal.lfilter([1.0], [1.0, -a], e)


def overlap_corr(w, shift):
    """重なった2区間のピリオドグラムの相関（白色雑音のとき） c = (Σ w[n] w[n+D])² / (Σ w²)²"""
    return (np.sum(w[:-shift] * w[shift:]) / np.sum(w ** 2)) ** 2


def main():
    a = 0.9
    print("[1] 自前のウェルチ法と scipy.signal.welch（L = 512、重なり 256、ハン窓）")
    x = ar1(2 ** 16, a)
    f, P_own, K = welch(x, FS, 512)
    f2, P_sp = signal.welch(x, FS, window="hann", nperseg=512)
    rel = np.max(np.abs(P_own - P_sp) / P_sp)
    print(f"  区間の数 K = {K}、最大の相対誤差 {rel:.1e}（許容 1e-10）")
    assert rel < 1e-10 and np.allclose(f, f2)

    print("\n[2] ウィーナー–ヒンチン（有限版）：|DFT|²/N と、標本自己相関 r̂[k] の DFT は同じ")
    N = 1024
    xs = x[:N]
    I = np.abs(np.fft.fft(xs, 2 * N)) ** 2 / N                     # 長さ 2N でゼロ詰め（巡回の重なりを防ぐ）
    r = np.correlate(xs, xs, mode="full") / N                      # r̂[k] = (1/N) Σ x[n+k] x[n]、k = −(N−1)…N−1
    r_circ = np.r_[r[N - 1:], 0.0, r[:N - 1]]                      # k = 0…N−1, (N), −(N−1)…−1 の順に並べる
    print(f"  max|ピリオドグラム − r̂ の DFT| / max = {np.max(np.abs(I - np.fft.fft(r_circ).real)) / I.max():.1e}")
    print(f"  r̂[0..3] = {np.round(r[N - 1:N + 3], 2)}、理論 a^k/(1−a²) = {np.round(a ** np.arange(4) / (1 - a ** 2), 2)}"
          "（N = 1024 では一致は大まか）")

    print("\n[3] AR(1)（a = 0.9）の理論の片側 PSD 2/(fs|1 − a e^{−jω}|²) と推定")
    S_th = lambda ff: 2 / (FS * np.abs(1 - a * np.exp(-2j * np.pi * ff / FS)) ** 2)
    for ff in (500, 2000, 6000):
        i = np.argmin(np.abs(f - ff))
        print(f"  {f[i]:6.0f} Hz：理論 {10*np.log10(S_th(f[i])):6.2f} dB、ウェルチ {10*np.log10(P_own[i]):6.2f} dB")
    print(f"  0 Hz と 8000 Hz の理論の比 {10*np.log10(S_th(0) / S_th(8000)):.1f} dB（(1+a)²/(1−a)² = 361 倍）")

    print("\n[4] 白色雑音（分散 1、片側 PSD の真の値 2/fs）：ピリオドグラムの ばらつき（標準偏差/平均、中ほどのビン）")
    S_true = 2 / FS
    for N in (1024, 16384, 262144):
        _, P = periodogram(RNG.standard_normal(N), FS)
        q = P[1:-1] / S_true
        print(f"  N = {N:6d}：平均 {q.mean():.3f}、標準偏差/平均 {q.std() / q.mean():.3f}（長くしても 1 のまま）")
    N, L = 16384, 512
    w = signal.get_window("hann", L)
    c = overlap_corr(w, L // 2)
    reps = np.array([welch(RNG.standard_normal(N), FS, L)[1][1:-1] / S_true for _ in range(400)])
    K = (N - L) // (L // 2) + 1
    nu = 2 * K / (1 + 2 * c)
    print(f"  ウェルチ法 N = {N}、L = {L}：K = {K}、重なりの相関 c = {c:.4f}、"
          f"標準偏差/平均 実測 {reps.std() / reps.mean():.3f}、理論 √((1+2c)/K) = {np.sqrt((1 + 2 * c) / K):.3f}")
    lo, hi = nu / stats.chi2.ppf(0.975, nu), nu / stats.chi2.ppf(0.025, nu)
    cover = np.mean((reps * lo <= 1) & (1 <= reps * hi))
    print(f"  等価な自由度 ν ≈ 2K/(1+2c) = {nu:.1f}、95% 区間 [推定 × {lo:.3f}, 推定 × {hi:.3f}]"
          f"（{10*np.log10(lo):+.2f} 〜 {10*np.log10(hi):+.2f} dB）、真の値を含んだ割合 {cover:.3f}")
    e_lo, e_hi = -np.log(0.975), -np.log(0.025)
    print(f"  ピリオドグラム1本の値の 95% 範囲（指数分布）：真の値の {e_lo:.4f}〜{e_hi:.3f} 倍"
          f"（{10*np.log10(e_lo):+.1f} 〜 {10*np.log10(e_hi):+.1f} dB）")

    print("\n[5] 誤用1：白色雑音のピリオドグラム（N = 16384、内側のビン 8191 本）の最大のピーク")
    M = 8191
    peaks = [10 * np.log10(np.max(periodogram(RNG.standard_normal(16384), FS)[1][1:-1]) / S_true) for _ in range(200)]
    print(f"  平らな真の PSD より 平均 +{np.mean(peaks):.1f} dB 高いピークが「見える」"
          f"（理論 10log10(ln M + γ) = +{10*np.log10(np.log(M) + EULER):.1f} dB）")

    print("\n[6] 誤用2：AR(1)（a = 0.9）の 1000 点の平均の分散")
    n, R = 1000, 4000
    means = np.array([ar1(n, a).mean() for _ in range(R)])
    var_x = 1 / (1 - a ** 2)
    k = np.arange(1, n)
    exact = var_x / n * (1 + 2 * np.sum((1 - k / n) * a ** k))
    print(f"  実測 {means.var():.4f}、正しい式 {exact:.4f}、独立とみなした σ²/n = {var_x / n:.5f}"
          f"（{means.var() / (var_x / n):.1f} 倍の過小評価。(1+a)/(1−a) = {(1 + a) / (1 - a):.0f}）")
    print(f"  有効サンプル数 n(1−a)/(1+a) ≈ {n * (1 - a) / (1 + a):.0f}")

    print("\n[7] 誤解：dB にしてから平均する")
    q = reps[:, 100]
    print(f"  ウェルチ法の値（K = {K}）：dB の平均 {np.mean(10*np.log10(q)):+.3f} dB、平均の dB {10*np.log10(q.mean()):+.3f} dB")
    q1 = np.concatenate([periodogram(RNG.standard_normal(16384), FS)[1][1:-1] / S_true for _ in range(20)])
    print(f"  ピリオドグラム1本：dB の平均 {np.mean(10*np.log10(q1)):+.2f} dB（理論 −10γ/ln10 = {-10 * EULER / np.log(10):+.2f} dB）")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    fN, PN = periodogram(x[:16384], FS)
    ax[0].plot(fN, 10 * np.log10(PN), lw=0.4, color="0.6", label="ピリオドグラム（N = 16384）")
    ax[0].plot(f, 10 * np.log10(P_own), lw=1.5, label=f"ウェルチ法（L = 512、K = {(len(x) - 512) // 256 + 1}）")
    ax[0].plot(f, 10 * np.log10(S_th(f)), "k--", lw=1, label="理論 2/(fs|1 − a e^{−jω}|²)")
    ax[0].set(xlabel="周波数 [Hz]", ylabel="PSD [dB（1/Hz）]", title="AR(1)（a = 0.9）の PSD")
    ax[0].legend(fontsize=8)
    qq = np.linspace(0, 6, 300)
    ax[1].hist(q1, bins=120, range=(0, 6), density=True, alpha=0.6, label="ピリオドグラム / 真の値")
    ax[1].plot(qq, np.exp(-qq), "k--", label="指数分布 e^{−q}")
    ax[1].hist(reps.ravel(), bins=120, range=(0, 6), density=True, alpha=0.6, label="ウェルチ法 / 真の値")
    ax[1].set(xlabel="推定値 / 真の値", ylabel="密度", title="白色雑音：推定値の分布")
    ax[1].legend(fontsize=8)
    for a_ in ax:
        a_.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("psd.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("\npsd.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"最大の相対誤差 ([\d.e+-]+)") < 1e-10, "[1] 自前のウェルチ法と scipy の相対誤差が 1e-10 未満（教科書 1.8e-15）")
ok(num(r"/ max = ([\d.e+-]+)") < 1e-13, "[2] ウィーナー–ヒンチン（有限版）の差が 1e-15 程度")
sm = [float(v) for v in re.findall(r"標準偏差/平均 ([\d.]+)（長くしても", cap.stdout)]
ok(len(sm) == 3 and all(0.9 < v < 1.1 for v in sm), f"[4] ピリオドグラムの標準偏差/平均が N によらず 1.0 前後 {sm}")
ok(abs(num(r"標準偏差/平均 実測 ([\d.]+)") - 0.13) < 0.01 and abs(num(r"理論 √\(\(1\+2c\)/K\) = ([\d.]+)") - 0.129) < 0.001, "[4] ウェルチ法の標準偏差/平均 0.13 前後（理論 0.129）")
ok(abs(num(r"真の値を含んだ割合 ([\d.]+)") - 0.95) < 0.02, "[4] 区間が真の値を含んだ割合が 0.95 前後")
ok(abs(num(r"平均 \+([\d.]+) dB 高いピーク") - 9.7) < 0.3, "[5] 白色雑音のピリオドグラムの最大のピーク +9.7 dB 前後")
ok(abs(num(r"（([\d.]+) 倍の過小評価") - 19) < 2, "[6] 独立とみなすと約 19 倍の過小評価")
ok(abs(num(r"ピリオドグラム1本：dB の平均 ([-+\d.]+) dB") + 2.5) < 0.1, "[7] ピリオドグラム1本の dB の平均が約 −2.5 dB")

### 手順 C の 6：区間の長さを変える（ノートブック版で足したセル）

教科書では `main()` の [4] の `L` を 128 と 2048 にして実行します。ここでは、上で定義した `welch` を使って、[4] と同じ計算を $L$ = 128・512・2048 でくり返します（数秒）。

In [ ]:
# ノートブック版で足したセル：psd.py の [4] のウェルチ法の部分を、L を変えてくり返す（上の welch・overlap_corr・FS を使う）
from scipy import signal
rng = np.random.default_rng(1)
N, S_true = 16384, 2 / FS
res_L = {}
for L in (128, 512, 2048):
    reps = np.array([welch(rng.standard_normal(N), FS, L)[1][1:-1] / S_true for _ in range(200)])
    K = (N - L) // (L // 2) + 1
    c = overlap_corr(signal.get_window("hann", L), L // 2)
    res_L[L] = reps.std() / reps.mean()
    print(f"L = {L:4d}：K = {K:3d}、標準偏差/平均 実測 {res_L[L]:.3f}、理論 √((1+2c)/K) = {np.sqrt((1 + 2 * c) / K):.3f}、周波数の点の間隔 {FS / L:.1f} Hz")
ok(abs(res_L[128] - 0.065) < 0.01, "手順 C の 6：L = 128（K = 255）で約 0.065")
ok(abs(res_L[2048] - 0.27) < 0.03, "手順 C の 6：L = 2048（K = 15）で約 0.27")

## 3. 混合正規分布の EM と sklearn の照合（IV-4-3-4・3-8、手順 D の 7、`em_gmm.py`）

数秒です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-4：混合ガウス分布の EM アルゴリズムを自分で実装し、scikit-learn の GaussianMixture と照合する

使い方: cd ~/home-lab/partIV/iv4 && uv run python em_gmm.py
やること
  1. 1次元の混合ガウス（3成分）のデータに、自前の EM と sklearn を同じ初期値から同じ回数だけ走らせて比べる
  2. 対数尤度が反復で減らないこと、EM が補助関数法（下から押さえる版）であること（下界が今の値で接する）を確かめる
  3. 誤用：成分の数 K を対数尤度だけで比べる（K を増やすほど必ず大きくなる）→ BIC で比べる
  4. 最尤推定の落とし穴：1つの成分が1点につぶれると、尤度はいくらでも大きくなる
"""
import warnings

import numpy as np
from scipy import stats
from scipy.special import logsumexp
from sklearn.exceptions import ConvergenceWarning
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # tol = 0 で回数を固定するので「収束していない」と言われる

RNG = np.random.default_rng(2)


def log_joint(x, pi, mu, var):
    """log π_k + log N(x_i | μ_k, σ_k²)。形は (n, K)"""
    return np.log(pi)[None, :] + stats.norm.logpdf(x[:, None], mu[None, :], np.sqrt(var)[None, :])


def em_gmm(x, pi, mu, var, n_iter=50):
    """1次元の混合ガウスの EM。戻り値：pi, mu, var, 対数尤度の履歴, 下界のチェックの履歴"""
    pi, mu, var = pi.copy(), mu.copy(), var.copy()
    ll_hist, gap_hist = [], []
    for _ in range(n_iter):
        lj = log_joint(x, pi, mu, var)
        ll = logsumexp(lj, axis=1).sum()                       # 対数尤度 ℓ(θ')
        g = np.exp(lj - logsumexp(lj, axis=1, keepdims=True))  # E ステップ：負担率 γ_ik = p(z_i = k | x_i, θ')
        nk = g.sum(axis=0)                                     # M ステップ：重みつきの平均と分散
        pi = nk / len(x)
        mu = (g * x[:, None]).sum(axis=0) / nk
        var = (g * (x[:, None] - mu[None, :]) ** 2).sum(axis=0) / nk
        # 下界 F(q, θ) = Σ_i Σ_k γ_ik log(p(x_i, z_i = k | θ)/γ_ik)。θ' では ℓ(θ') に等しく、新しい θ では ℓ(θ) 以下
        lj_new = log_joint(x, pi, mu, var)
        with np.errstate(divide="ignore", invalid="ignore"):
            ent = -np.sum(np.where(g > 0, g * np.log(g), 0.0))
        F_old = np.sum(g * lj) + ent
        F_new = np.sum(g * lj_new) + ent
        ll_new = logsumexp(lj_new, axis=1).sum()
        ll_hist.append(ll)
        gap_hist.append((F_old - ll, ll_new - F_new, F_new - F_old))   # ≈0、≥0、≥0 になるはず
    ll_hist.append(logsumexp(log_joint(x, pi, mu, var), axis=1).sum())
    return pi, mu, var, np.array(ll_hist), np.array(gap_hist)


def main():
    true = dict(pi=np.array([0.5, 0.3, 0.2]), mu=np.array([-2.0, 1.0, 4.0]), sd=np.array([1.0, 0.5, 1.5]))
    n = 1500
    z = RNG.choice(3, n, p=true["pi"])
    x = RNG.normal(true["mu"][z], true["sd"][z])

    pi0, mu0, var0 = np.ones(3) / 3, np.array([-1.0, 0.0, 1.0]), np.ones(3)
    n_iter = 50
    pi, mu, var, ll, gap = em_gmm(x, pi0, mu0, var0, n_iter)
    print("[1] 自前の EM（50 回）")
    print(f"  重み {np.round(pi, 3)}、平均 {np.round(mu, 3)}、標準偏差 {np.round(np.sqrt(var), 3)}")
    print(f"  （真の値：重み {true['pi']}、平均 {true['mu']}、標準偏差 {true['sd']}）")

    gm = GaussianMixture(3, covariance_type="full", weights_init=pi0, means_init=mu0[:, None],
                         precisions_init=(1 / var0)[:, None, None], max_iter=n_iter, tol=0.0, reg_covar=0.0,
                         random_state=0)
    gm.fit(x[:, None])
    o = np.argsort(gm.means_[:, 0])
    print("  sklearn GaussianMixture（同じ初期値・同じ回数、reg_covar = 0、tol = 0）")
    d = max(np.max(np.abs(gm.weights_[o] - pi)), np.max(np.abs(gm.means_[o, 0] - mu)),
            np.max(np.abs(gm.covariances_[o, 0, 0] - var)))
    print(f"  パラメータの最大の差 {d:.1e}（許容 1e-8）、1点あたりの対数尤度 自前 {ll[-1] / n:.8f}"
          f"、sklearn score {gm.score(x[:, None]):.8f}")
    assert d < 1e-8

    print("\n[2] 対数尤度は減らない／下界は今の値で接する（EM ＝ 補助関数法）")
    print(f"  1 回目 {ll[0]:.2f} → 50 回目 {ll[-1]:.2f}、増分の最小 {np.min(np.diff(ll)):.2e}（0 以上）")
    print(f"  max|F(q', θ') − ℓ(θ')| = {np.max(np.abs(gap[:, 0])):.1e}（接する）、"
          f"min(ℓ(θ) − F(q', θ)) = {np.min(gap[:, 1]):.1e}（下界、0 以上）、"
          f"min(F(q', θ) − F(q', θ')) = {np.min(gap[:, 2]):.1e}（M ステップで下界が増える）")
    assert np.min(np.diff(ll)) > -1e-8

    print("\n[3] 誤用：成分の数 K を対数尤度だけで選ぶ")
    print("   K  対数尤度    パラメータ数  BIC（自前）  BIC（sklearn）")
    for K in range(1, 7):
        best = max((GaussianMixture(K, n_init=1, random_state=s).fit(x[:, None]) for s in range(5)),
                   key=lambda g_: g_.score(x[:, None]))
        llK = best.score(x[:, None]) * n
        p = 3 * K - 1                                  # 平均 K、分散 K、重み K−1
        bic = -2 * llK + p * np.log(n)
        print(f"  {K:2d}  {llK:9.2f}  {p:6d}      {bic:9.2f}   {best.bic(x[:, None]):9.2f}")
    print("  対数尤度は K を増やすほど（ほぼ）増える。BIC の最小は真の K = 3 の近く")

    print("\n[4] 最尤推定の落とし穴：4つ目の成分（重み 0.01）を1点 x₁ につぶすと、尤度は上限なしに大きくなる")
    print(f"  EM の3成分の解：対数尤度 {ll[-1]:.2f}")
    for s in (1e-2, 1e-8, 1e-16, 1e-50):
        lj = log_joint(x, np.r_[0.99 * pi, 0.01], np.r_[mu, x[0]], np.r_[var, s ** 2])
        print(f"  4つ目の成分の標準偏差 {s:g}：対数尤度 {logsumexp(lj, axis=1).sum():.2f}")
    print("  → 分散に下限を置く（sklearn の reg_covar）、事前分布を置く（MAP）などで防ぐ")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"パラメータの最大の差 ([\d.e+-]+)") < 1e-8, "[1] sklearn とのパラメータの最大の差が 1e-8 未満（教科書 3e-15）")
m = re.search(r"自前 ([-\d.]+)、sklearn score ([-\d.]+)", cap.stdout)
ok(abs(float(m.group(1)) - float(m.group(2))) < 2e-8, f"[1] 1点あたりの対数尤度が小数8桁でほぼ同じ（{m.group(1)}、{m.group(2)}。差が 2e-8 未満）")
ok(num(r"増分の最小 ([-\d.e+]+)") >= 0, "[2] 対数尤度の増分の最小が 0 以上")
bic = {int(k): float(b) for k, b in re.findall(r"^\s+(\d)\s+[-\d.]+\s+\d+\s+([\d.]+)\s+[\d.]+$", cap.stdout, flags=re.M)}
ok(len(bic) == 6 and min(bic, key=bic.get) == 3, f"[3] BIC の最小は K = 3（{bic}）")
ok(num(r"標準偏差 1e-50：対数尤度 ([-\d.]+)") > num(r"EM の3成分の解：対数尤度 ([-\d.]+)"), "[4] 標準偏差を小さくすると、対数尤度が EM の解を超える")

## 4. 補助関数法の3つの例（IV-4-3-8、手順 D の 8、`mm_demo.py`）

ラプラス分布の位置（中央値）、t 分布の位置と尺度、IS-NMF の1回の更新を、補助関数法で解いて照合します（数秒）。どこかで `AssertionError` が出たら、写し間違いを探します。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-4：補助関数法（MM アルゴリズム）の小さな例を3つ解き、既存のライブラリと照合する

使い方: cd ~/home-lab/partIV/iv4 && uv run python mm_demo.py
  A. ラプラス分布の位置（Σ|x_i − μ| の最小 ＝ 中央値）を、|u| の2次の上界で解く（重みつき最小二乗のくり返し）
     → np.median と照合
  B. t 分布（自由度 ν を固定）の位置と尺度の最尤推定を、log の接線で解く（EM と同じ式になる）
     → scipy.stats.t.fit(x, fix_df=ν) と照合
  C. IS-NMF の補助関数を実際に計算し、「どこでも J 以上」「今の値で等しい」「更新式が補助関数の最小」を確かめる。
     1 回の更新を scikit-learn の NMF（solver="mu", beta_loss="itakura-saito"）の 1 回と照合
"""
import warnings

import numpy as np
from scipy import stats
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # max_iter=1 で止めるので出る警告
RNG = np.random.default_rng(3)
EPS = 1e-12


def laplace_location_mm(x, mu=0.0, n_iter=100):
    """J(μ) = Σ|x_i − μ| を、|u| ≤ u²/(2|u'|) + |u'|/2 で上から押さえて最小にする"""
    hist = []
    for _ in range(n_iter):
        hist.append(np.sum(np.abs(x - mu)))
        w = 1.0 / np.maximum(np.abs(x - mu), 1e-12)   # 補助関数は Σ w_i (x_i − μ)²/2 + 定数
        mu = np.sum(w * x) / np.sum(w)                # その最小（重みつき平均）
    hist.append(np.sum(np.abs(x - mu)))
    return mu, np.array(hist)


def t_nll(x, mu, s2, nu):
    return -np.sum(stats.t.logpdf(x, nu, loc=mu, scale=np.sqrt(s2)))


def t_fit_mm(x, nu, n_iter=200):
    """t 分布の負の対数尤度（定数を除く）
       J = (n/2) log σ² + ((ν+1)/2) Σ log(1 + d_i²/(ν σ²))、d_i = x_i − μ
    log(1 + u) は u について凹なので、今の値 u' での接線で上から押さえると d_i² の重みつき和になる：
       w_i = (ν+1)/(ν + d_i'²/σ'²) → μ = Σ w x / Σ w、σ² = (1/n) Σ w (x − μ)²"""
    mu, s2 = np.median(x), np.var(x)
    hist = []
    for _ in range(n_iter):
        hist.append(t_nll(x, mu, s2, nu))
        w = (nu + 1) / (nu + (x - mu) ** 2 / s2)
        mu = np.sum(w * x) / np.sum(w)
        s2 = np.mean(w * (x - mu) ** 2)
    hist.append(t_nll(x, mu, s2, nu))
    return mu, s2, np.array(hist)


def is_cost(X, Y):
    r = X / Y
    return float(np.sum(r - np.log(r) - 1))


def is_aux(W, Wp, H, X):
    """IS-NMF の W についての補助関数 G(W|W')（定数を除いたコスト J(W) = Σ (x/y + log y) と比べる）
       1/y ≤ Σ_k λ_k²/(w_fk h_kt)（イェンセン、λ = w'h/y'）、log y ≤ log y' + (y − y')/y'（接線）"""
    Yp = Wp @ H
    lam = Wp[:, :, None] * H[None, :, :] / Yp[:, None, :]            # (F, K, T)
    term1 = np.sum(lam ** 2 * X[:, None, :] / (W[:, :, None] * H[None, :, :]))
    Y = W @ H
    term2 = np.sum(np.log(Yp) + (Y - Yp) / Yp)
    return term1 + term2


def is_J(W, H, X):
    Y = W @ H
    return float(np.sum(X / Y + np.log(Y)))


def main():
    print("[A] ラプラス分布の位置（中央値）を補助関数法で")
    x = RNG.laplace(1.0, 2.0, 101)
    mu, h = laplace_location_mm(x)
    print(f"  補助関数法 {mu:.10f}、np.median {np.median(x):.10f}、差 {abs(mu - np.median(x)):.1e}（許容 1e-6）")
    print(f"  コスト：最初 {h[0]:.4f} → 最後 {h[-1]:.4f}、1回で増えた最大 {np.max(np.diff(h)):.1e}（0 以下）")
    assert abs(mu - np.median(x)) < 1e-6 and np.max(np.diff(h)) <= 1e-9

    print("\n[B] t 分布（ν = 4 で固定）の位置と尺度")
    nu = 4.0
    x = stats.t.rvs(nu, loc=2.0, scale=1.5, size=2000, random_state=RNG)
    mu, s2, h = t_fit_mm(x, nu)
    df, loc, sc = stats.t.fit(x, fix_df=nu)
    print(f"  補助関数法 μ = {mu:.6f}、σ = {np.sqrt(s2):.6f}、負の対数尤度 {h[-1]:.6f}")
    print(f"  scipy      μ = {loc:.6f}、σ = {sc:.6f}、負の対数尤度 {t_nll(x, loc, sc**2, nu):.6f}")
    print(f"  差 μ {abs(mu - loc):.1e}、σ {abs(np.sqrt(s2) - sc):.1e}（許容 1e-4。scipy は汎用の最適化で止まり方が違う）")
    print(f"  1回で増えた最大 {np.max(np.diff(h)):.1e}（数値誤差の範囲で 0 以下）")
    assert abs(mu - loc) < 1e-4 and abs(np.sqrt(s2) - sc) < 1e-4 and np.max(np.diff(h)) <= 1e-9

    print("\n[C] IS-NMF の補助関数（F = 30、T = 40、K = 4）")
    F, T, K = 30, 40, 4
    X = (RNG.uniform(0.1, 1, (F, K)) @ RNG.uniform(0.1, 1, (K, T))) * RNG.exponential(1.0, (F, T))
    Wp = RNG.uniform(0.1, 1, (F, K))
    H = RNG.uniform(0.1, 1, (K, T))
    print(f"  今の値で等しい：|G(W'|W') − J(W')| = {abs(is_aux(Wp, Wp, H, X) - is_J(Wp, H, X)):.1e}")
    gaps = [is_aux(Wr, Wp, H, X) - is_J(Wr, H, X)
            for Wr in (Wp * RNG.uniform(*rg, (F, K)) for rg in ((0.2, 5), (0.99, 1.01)) for _ in range(500))]
    print(f"  ほかの W 1000 個（大きくずらす 500 個、1% だけずらす 500 個）で G − J の最小 {min(gaps):.3e}（0 以上 → どこでも J 以上）")
    Y = Wp @ H
    Wn = Wp * np.sqrt(((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T))           # 平方根つきの更新
    G_new = is_aux(Wn, Wp, H, X)
    G_pert = min(is_aux(Wn * RNG.uniform(0.95, 1.05, (F, K)), Wp, H, X) for _ in range(300))
    print(f"  更新後の W は補助関数の最小：G(W_new) = {G_new:.6f}、少しずらした W の G の最小 {G_pert:.6f}（大きい）")
    print(f"  J：{is_J(Wp, H, X):.6f} → {is_J(Wn, H, X):.6f}（下がる）")
    Wl = Wp * (((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T))                  # 平方根なし（ヒューリスティック）
    print(f"  平方根なしの更新：G = {is_aux(Wl, Wp, H, X):.6f}（G の最小ではないが、G(W'|W') = {is_aux(Wp, Wp, H, X):.6f} と同じ高さ）、"
          f"J = {is_J(Wl, H, X):.6f}")
    print("  　→ G は要素ごとに A/w + Bw の形で、平方根なしの点 w'' = w*²/w' では G(w'') = G(w') になる。"
          "J(w'') ≤ G(w'') = J(w') なので、IS では平方根なしでも増えない（ただし1回ごとの G の下がり方は平方根つきが最大）")
    assert min(gaps) > -1e-9 and G_pert > G_new

    # scikit-learn の NMF で 1 回（W を更新してから H を更新）
    Hp = H.copy()
    m = NMF(K, solver="mu", beta_loss="itakura-saito", init="custom", max_iter=1, tol=0.0)
    W_sk = m.fit_transform(X, W=Wp.copy(), H=Hp.copy())
    H_sk = m.components_
    W1 = Wp * np.sqrt(((X / Y ** 2) @ Hp.T) / ((1 / Y) @ Hp.T))
    Y1 = W1 @ Hp
    H1 = Hp * np.sqrt((W1.T @ (X / Y1 ** 2)) / (W1.T @ (1 / Y1)))
    d = max(np.max(np.abs(W_sk - W1) / W1), np.max(np.abs(H_sk - H1) / H1))
    print(f"  scikit-learn の 1 回と自前の 1 回の最大の相対差 {d:.1e}（許容 1e-6。sklearn は 0 割りを防ぐ小さな値を足す）")
    assert d < 1e-6


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"np\.median [\d.]+、差 ([\d.e+-]+)") < 1e-6 and num(r"\[A\][\s\S]*?1回で増えた最大 ([-\d.e+]+)") <= 1e-9, "[A] np.median との差が 1e-6 未満、1回で増えた最大が 0 以下")
ok(num(r"差 μ ([\d.e+-]+)") < 1e-4 and num(r"、σ ([\d.e+-]+)（許容 1e-4") < 1e-4, "[B] scipy との差が 1e-4 未満")
ok(num(r"\|G\(W'\|W'\) − J\(W'\)\| = ([\d.e+-]+)") < 1e-9, "[C] 今の値で G = J（差 0）")
ok(num(r"G − J の最小 ([-\d.e+]+)") >= 0, "[C] G − J の最小が 0 以上")
ok(num(r"最大の相対差 ([\d.e+-]+)（許容 1e-6") < 1e-6, "[C] sklearn との差が 1e-6 未満")

## 5. 音声の統計を見て、聞く（IV-4-3-6、手順 E、`speech_stats.py`）

初回は libri1 をダウンロードします（十数秒）。図の左で、データ（黒い点）が真ん中で尖り、裾が正規分布の放物線より大きく外へ広がることを見てください。自分の声で試すときは、`sys.argv` に wav のファイル名を足します（手順 E の 11）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['speech_stats.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-4：音声の STFT 係数の分布を描き、ガウス・ラプラス・t 分布と比べる（スーパーガウス性と尖度）

使い方: cd ~/home-lab/partIV/iv4 && uv run python speech_stats.py [音声の wav]
  wav を指定しないときは librosa の例の音源 libri1（英語の朗読。ダウンロードが必要）
  第7章で録った自分の声（../../ch7/data/my_voice.wav）も使える
やること
  1. 500〜4000 Hz のビンの係数を、ビンごとの RMS で割ってそろえ、実部・虚部を集めて尖度を求める
     （同じ長さ・同じ処理のガウス雑音と比べる）
  2. ガウス・ラプラス・t 分布を最尤推定で当てはめ、対数尤度と AIC を比べる（パラメータの数の違いに注意）
  3. 局所的な分散で割ると、ほぼガウスに戻ることを確かめる（「局所ガウスモデル」）
  4. 耳で確かめる wav を4つ作る（どれも同じ RMS）：
       ss_1_speech.wav       元の音声
       ss_2_white.wav        同じ RMS の白色ガウス雑音
       ss_3_stationary.wav   音声の長時間平均のスペクトルをもつ定常なガウス雑音（分散が時間で変わらない）
       ss_4_localgauss.wav   x_ft ~ N_C(0, v_ft)、v_ft は音声のパワーをならしたもの（分散だけが音声と同じ）
  図 speech_stats.png
"""
import pathlib
import sys

import numpy as np
import librosa
from scipy import stats
from scipy.ndimage import uniform_filter
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft, istft                                      # IV-2 の共通ライブラリ
from common.audio_io import load, save as save_wav                       # 同上（save は 1 を超える値を止める）

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(4)
FS, L, S = 16000, 512, 128
TARGET_RMS = 0.03                        # 約 −30 dBFS。聞くときはさらに Mac の音量を小さくしてから


def kurt(x):
    """尖度 E[(x−μ)⁴]/σ⁴（正規分布で 3。scipy の既定の「超過尖度」は 3 を引いた値）"""
    return stats.kurtosis(x, fisher=False)


def coeffs(X, f):
    """ビンをそろえる：ビンごとに RMS で割り、選んだビンの実部と虚部を1列に並べる"""
    B = X[f] / np.sqrt(np.mean(np.abs(X[f]) ** 2, axis=1, keepdims=True))
    return np.r_[B.real.ravel(), B.imag.ravel()], B


def fit_three(x):
    """ガウス・ラプラス・t（位置は 0 に固定）の最尤推定。戻り値：{名前: (対数尤度, パラメータ数, 分布)}"""
    out = {}
    s = np.sqrt(np.mean(x ** 2))
    out["ガウス"] = (np.sum(stats.norm.logpdf(x, 0, s)), 1, stats.norm(0, s))
    b = np.mean(np.abs(x))
    out["ラプラス"] = (np.sum(stats.laplace.logpdf(x, 0, b)), 1, stats.laplace(0, b))
    nu, _, sc = stats.t.fit(x, floc=0)
    out["t"] = (np.sum(stats.t.logpdf(x, nu, 0, sc)), 2, stats.t(nu, 0, sc))
    return out


def save(name, y):
    y = y * TARGET_RMS / np.sqrt(np.mean(y ** 2))
    if np.max(np.abs(y)) > 0.99:
        y = y * 0.99 / np.max(np.abs(y))
    save_wav(name, y, FS)


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else librosa.ex("libri1")
    y, _ = librosa.load(path, sr=FS, mono=True)
    y = y / np.sqrt(np.mean(y ** 2))
    X = stft(y, L, S)                                         # (F, T)
    freqs = np.arange(X.shape[0]) * FS / L
    f = (freqs >= 500) & (freqs <= 4000)
    print(f"音声 {len(y) / FS:.1f} 秒、STFT {X.shape}（L = {L}、S = {S}）、使うビン {f.sum()} 本（500〜4000 Hz）")

    noise = RNG.standard_normal(len(y))
    N = stft(noise, L, S)
    xs, Bs = coeffs(X, f)
    xn, _ = coeffs(N, f)
    print("\n[1] 尖度（正規分布なら 3）")
    print(f"  音声の STFT 係数（実部・虚部）{kurt(xs):6.2f}、ガウス雑音 {kurt(xn):5.2f}")
    print(f"  音声の波形そのもの {kurt(y):6.2f}、ガウス雑音の波形 {kurt(noise):5.2f}")
    p4 = np.mean(np.abs(Bs) ** 4) / np.mean(np.abs(Bs) ** 2) ** 2
    print(f"  複素の尖度 E|x|⁴/(E|x|²)²：音声 {p4:.2f}（複素正規分布なら 2）")

    print("\n[2] 分布の当てはめ（n は実部・虚部を合わせた数。隣のフレームは窓が重なり独立ではない）")
    sub = RNG.choice(len(xs), min(len(xs), 100_000), replace=False)   # t の当てはめの時間を抑えるため 10 万個
    for name, x in (("音声", xs[sub]), ("ガウス雑音", xn[sub])):
        res = fit_three(x)
        print(f"  {name}（n = {len(x)}）")
        aics = {k: -2 * v[0] + 2 * v[1] for k, v in res.items()}
        for k, (ll, p, dist) in res.items():
            extra = f"、ν = {dist.args[0]:.2f}" if k == "t" else ""
            print(f"    {k:5s} 対数尤度 {ll:12.1f}  パラメータ {p}  AIC {aics[k]:12.1f}{extra}")
        print(f"    → AIC で一番よいのは {min(aics, key=aics.get)}")
        if name == "音声":
            fits_s = res
        else:
            fits_n = res

    print("\n[3] 局所的な分散で割る（周りの 3 ビン × 5 フレームのパワーの平均。真ん中は除く）")
    P = np.abs(X) ** 2
    k_f, k_t = 3, 5
    loc = (uniform_filter(P, size=(k_f, k_t), mode="nearest") * k_f * k_t - P) / (k_f * k_t - 1)
    Z = X[f] / np.sqrt(loc[f] + 1e-12)
    zz = np.r_[Z.real.ravel(), Z.imag.ravel()]
    zz = zz / np.sqrt(np.mean(zz ** 2))
    print(f"  尖度 {kurt(xs):.2f} → {kurt(zz):.2f}（3 に近づく：音声は「分散が時間と周波数で変わるガウス」に近い）")

    print("\n[4] 耳で確かめる wav（どれも RMS をそろえた。最初は小さな音量で）")
    save("ss_1_speech.wav", y)
    save("ss_2_white.wav", noise)
    cn = lambda v: np.sqrt(v / 2) * (RNG.standard_normal(v.shape) + 1j * RNG.standard_normal(v.shape))
    v_stat = np.repeat(P.mean(axis=1, keepdims=True), P.shape[1], axis=1)
    v_loc = uniform_filter(P, size=(3, 3), mode="nearest")
    for name, v in (("ss_3_stationary.wav", v_stat), ("ss_4_localgauss.wav", v_loc)):
        Xg = cn(v)
        Xg[0] = Xg[0].real                                       # 直流とナイキストは実数にする
        Xg[-1] = Xg[-1].real
        save(name, istft(Xg, L, S, "hann", length=len(y)))
    for name in ("ss_1_speech.wav", "ss_2_white.wav", "ss_3_stationary.wav", "ss_4_localgauss.wav"):
        z, _ = load(name)
        print(f"  {name:22s} 波形の尖度 {kurt(z):6.2f}、STFT 係数の尖度 {kurt(coeffs(stft(z, L, S), f)[0]):6.2f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    grid = np.linspace(-8, 8, 400)
    for a, x, res, title in ((ax[0], xs, fits_s, "音声の STFT 係数（実部・虚部）"), (ax[1], xn, fits_n, "ガウス雑音の STFT 係数")):
        h, e = np.histogram(x, bins=161, range=(-8, 8), density=True)
        a.semilogy((e[1:] + e[:-1]) / 2, h, ".", ms=3, color="k", label="データ")
        for k, (ll, p, dist) in res.items():
            a.semilogy(grid, dist.pdf(grid), label=k)
        a.set(title=title, xlabel="値（ビンごとの RMS で割った）", ylim=(1e-6, 2))
        a.grid(True, alpha=0.3)
        a.legend(fontsize=8)
    ax[0].set_ylabel("密度（対数目盛り）")
    fig.tight_layout()
    fig.savefig("speech_stats.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("\nspeech_stats.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"音声の STFT 係数（実部・虚部）\s*([\d.]+)") > 20 and abs(num(r"ガウス雑音\s+([\d.]+)\n") - 3) < 0.1, "[1] 音声の STFT 係数の尖度が 3 よりずっと大きい（教科書 59）、ガウス雑音が 3.0 前後")
sp, ns = cap.stdout.split("ガウス雑音（n")[0].split("[2]")[1], cap.stdout.split("ガウス雑音（n")[1].split("[3]")[0]
aic = lambda part, k: num(rf"{k}\s+対数尤度.*?AIC\s+([\d.]+)", text=part)
ok(aic(sp, "t") < aic(sp, "ラプラス") < aic(sp, "ガウス"), "[2] 音声で AIC が「t ＜ ラプラス ＜ ガウス」の順")
ok("一番よいのは ガウス" in ns and 0 < aic(ns, "t") - aic(ns, "ガウス") <= 2.5, f"[2] ガウス雑音ではガウスが選ばれる（t の AIC は {aic(ns, 't') - aic(ns, 'ガウス'):.1f} 悪い。教科書 1.4）")
ok(abs(num(r"尖度 [\d.]+ → ([\d.]+)") - 3) < 0.5, "[3] 局所的な分散で割ると尖度が 3 前後（教科書 2.82）")
ok(num(r"ss_4_localgauss\.wav.*?STFT 係数の尖度\s+([\d.]+)") > 20 and abs(num(r"ss_3_stationary\.wav.*?STFT 係数の尖度\s+([\d.]+)") - 3) < 0.2, "[4] 4 の STFT 係数の尖度が音声と同じくらい大きく（教科書 55）、3 は 3 前後")

4つの音を聞き比べます（手順 E の 10）。3 は時間で変わらない「サー」という雑音です。4 で、ことばのリズム（音節の区切り）や声の高さの動きがどのくらい聞こえるかをノートに書いてください。`ss_2_white.wav` は最後に、特に小さく。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['ss_1_speech.wav', 'ss_3_stationary.wav', 'ss_4_localgauss.wav', 'ss_2_white.wav']:
    print(f)
    listen_file(f)

## 6. ウィーナーフィルタと時間周波数マスク（IV-4-3-5、手順 F、`wiener.py`）

入力の SNR 0 dB（音声の区間）で実行します。図で、理想マスクは声のある場所だけが白く、推定マスク（DD）はそれより暗く、まだらになっていることを見てください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['wiener.py', '--snr', '0']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-4：ウィーナーフィルタ。時間領域の FIR 版（ウィーナー–ホップ方程式）と、STFT 上のマスク版を作り、聞き比べる

使い方: cd ~/home-lab/partIV/iv4 && uv run python wiener.py [音声の wav] [--snr 0]
  wav を指定しないときは librosa の例の音源 libri1（ダウンロードが必要）
やること
  0. 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1) のとき、E[s|x] = 0.8x、誤差の分散 0.8 を乱数で確かめる
  1. 音声の前に 1 秒の「雑音だけ」の区間をつけ、定常な雑音（色つき）を SNR 0 dB で足す
  2. 時間領域の FIR ウィーナーフィルタ（32 タップ）：ウィーナー–ホップ方程式 R h = r を
     自前（テプリッツ行列を作って np.linalg.solve）と scipy.linalg.solve_toeplitz で解いて照合。
     r = R_x − R_n（音声と雑音は無相関）なので、雑音だけの区間から R_n を推定すれば、正解の音声なしで作れる
  3. STFT のマスク：理想ウィーナーマスク（正解を使う。上限）、理想バイナリマスク、
     推定マスク（雑音の分散は最初の 1 秒、音声の分散は ML 推定 max(|x|² − λ_n, 0)、または decision-directed）
  4. 出力の SNR を比べ、wav に書き出す（wn_*.wav、どれも同じ倍率で。最初は小さな音量で聞く）。図 wiener.png
"""
import argparse
import pathlib
import sys

import numpy as np
import librosa
from scipy import linalg, signal
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft, istft                                      # IV-2 の共通ライブラリ
from common.audio_io import save                                         # 同上（1 を超える値は止める）

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(5)
FS, L, S = 16000, 512, 128
LEAD = FS                                          # 最初の 1 秒は雑音だけ
EPS = 1e-12


def snr_db(ref, est):
    """出力の SNR = 10 log10(Σ s² / Σ (s − ŝ)²)（尺度もそろっていることを求める、きびしい指標）"""
    return 10 * np.log10(np.sum(ref ** 2) / np.sum((ref - est) ** 2))


def autocorr(x, p):
    """標本自己相関 r̂[k] = (1/N) Σ x[n] x[n−k]、k = 0…p−1"""
    N = len(x)
    return np.array([np.dot(x[k:], x[:N - k]) / N for k in range(p)])


def fir_wiener(x, r_sx, p):
    """ウィーナー–ホップ方程式 Σ_k h[k] R_x[l−k] = r_sx[l]（l = 0…p−1）を自前で解く"""
    r = autocorr(x, p)
    R = np.array([[r[abs(i - j)] for j in range(p)] for i in range(p)])   # テプリッツ行列
    return np.linalg.solve(R, r_sx), r


def dd_mask(X, lam_n, alpha=0.98):
    """decision-directed 法（Ephraim–Malah）で事前 SNR ξ を推定し、ウィーナーゲイン ξ/(1+ξ) を返す"""
    F, T = X.shape
    G = np.zeros((F, T))
    prev = np.zeros(F)                                  # 1 つ前のフレームの |ŝ|²
    for t in range(T):
        gamma = np.abs(X[:, t]) ** 2 / lam_n            # 事後 SNR
        xi = alpha * prev / lam_n + (1 - alpha) * np.maximum(gamma - 1, 0)
        G[:, t] = xi / (1 + xi)
        prev = (G[:, t] * np.abs(X[:, t])) ** 2
    return G


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("wav", nargs="?")
    ap.add_argument("--snr", type=float, default=0.0)
    a = ap.parse_args()

    print("[0] 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1)、x = s + n")
    n0 = 200_000
    cn = lambda v, n: np.sqrt(v / 2) * (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
    s0, x0 = cn(4.0, n0), None
    x0 = s0 + cn(1.0, n0)
    g = np.sum(s0 * x0.conj()) / np.sum(np.abs(x0) ** 2)
    print(f"  最小二乗の係数 {g.real:.4f}{g.imag:+.4f}j（理論 λ_s/(λ_s+λ_n) = 0.8）、"
          f"誤差の分散 {np.mean(np.abs(s0 - 0.8 * x0) ** 2):.4f}（理論 λ_sλ_n/(λ_s+λ_n) = 0.8）")

    path = a.wav or librosa.ex("libri1")
    sp, _ = librosa.load(path, sr=FS, mono=True)
    s = np.r_[np.zeros(LEAD), sp / np.sqrt(np.mean(sp ** 2))]
    noise = signal.lfilter([1.0], [1.0, -0.7], RNG.standard_normal(len(s)))   # 定常な色つき雑音（AR(1)）
    act = s[LEAD:]
    noise *= np.sqrt(np.mean(act ** 2) / np.mean(noise ** 2)) / 10 ** (a.snr / 20)
    x = s + noise
    print(f"\n[1] {len(sp) / FS:.1f} 秒の音声＋1 秒の雑音だけの区間、入力の SNR {snr_db(s, x):.2f} dB（音声の区間で {a.snr} dB）")

    print("\n[2] 時間領域の FIR ウィーナーフィルタ（p = 32）")
    p = 32
    r_n = autocorr(noise[:LEAD], p)                    # 雑音だけの区間から R_n を推定
    r_x = autocorr(x, p)
    h_own, _ = fir_wiener(x, r_x - r_n, p)             # r_sx = R_x − R_n（無相関なので）
    h_sp = linalg.solve_toeplitz(r_x, r_x - r_n)
    print(f"  自前（np.linalg.solve）と scipy.linalg.solve_toeplitz の最大の差 {np.max(np.abs(h_own - h_sp)):.1e}（許容 1e-8）")
    assert np.max(np.abs(h_own - h_sp)) < 1e-8
    r_sx_oracle = np.array([np.dot(s[k:], x[:len(x) - k]) / len(x) for k in range(p)])   # 正解を使う版
    h_or, _ = fir_wiener(x, r_sx_oracle, p)
    y_fir = signal.lfilter(h_own, [1.0], x)
    y_or = signal.lfilter(h_or, [1.0], x)
    print(f"  出力の SNR：R_n を推定した版 {snr_db(s, y_fir):.2f} dB、正解の相互相関を使った版 {snr_db(s, y_or):.2f} dB")

    print("\n[3] STFT のマスク（L = 512、S = 128）")
    X, Sx, Nx = stft(x, L, S), stft(s, L, S), stft(noise, L, S)
    lam_n = np.mean(np.abs(X[:, : LEAD // S - L // S]) ** 2, axis=1)        # 最初の 1 秒のフレーム（音声と重ならない分）
    masks = {
        "ideal_wiener": np.abs(Sx) ** 2 / (np.abs(Sx) ** 2 + np.abs(Nx) ** 2 + EPS),
        "ideal_binary": (np.abs(Sx) > np.abs(Nx)).astype(float),
    }
    lam_s_ml = np.maximum(np.abs(X) ** 2 - lam_n[:, None], 0)                # 1 つの係数からの ML 推定
    masks["ml"] = lam_s_ml / (lam_s_ml + lam_n[:, None])
    masks["dd"] = dd_mask(X, lam_n[:, None][:, 0])
    out = {"noisy": x}
    for k, Mk in masks.items():
        out[k] = istft(Mk * X, L, S, "hann", length=len(x))
    out["fir"] = y_fir
    labels = {"noisy": "雑音入り（処理なし）", "fir": "FIR ウィーナー（R_n を推定）",
              "ideal_wiener": "理想ウィーナーマスク（正解を使う）", "ideal_binary": "理想バイナリマスク（正解を使う）",
              "ml": "推定マスク（ML、1 係数）", "dd": "推定マスク（decision-directed）"}
    scale = 0.03 / np.sqrt(np.mean(x ** 2))             # すべて同じ倍率で書き出す（大きさの違いも聞こえるように）
    res = {}
    for k, yk in out.items():
        res[k] = snr_db(s, yk)
        save(f"wn_{k}.wav", np.clip(yk * scale, -0.99, 0.99), FS)
        print(f"  {labels[k]:28s} SNR {res[k]:6.2f} dB → wn_{k}.wav")
    save("wn_clean.wav", np.clip(s * scale, -0.99, 0.99), FS)
    print("  （正解の音声は wn_clean.wav）")

    fig, ax = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
    ext = [0, X.shape[1] * S / FS, 0, FS / 2]
    db = lambda Z: 10 * np.log10(np.abs(Z) ** 2 + 1e-10)
    vmax = db(X).max()
    for a_, Z, t in ((ax[0, 0], X, "雑音入り |x|²（dB）"), (ax[0, 1], masks["dd"] * X, "推定マスク（DD）の出力（dB）")):
        a_.imshow(db(Z), origin="lower", aspect="auto", extent=ext, vmin=vmax - 70, vmax=vmax, cmap="magma")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    for a_, Mk, t in ((ax[1, 0], masks["ideal_wiener"], "理想ウィーナーマスク"), (ax[1, 1], masks["dd"], "推定マスク（DD）")):
        im = a_.imshow(Mk, origin="lower", aspect="auto", extent=ext, vmin=0, vmax=1, cmap="gray")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    fig.colorbar(im, ax=ax[1, :], shrink=0.8, label="ゲイン")
    fig.savefig("wiener.png", dpi=110)
    plt.show()   # ノートブック版：図を表示する
    print("\nwiener.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(abs(num(r"最小二乗の係数 ([\d.]+)") - 0.8) < 0.01 and abs(num(r"誤差の分散 ([\d.]+)") - 0.8) < 0.01, "[0] 係数が 0.80 前後、誤差の分散が 0.80 前後")
ok(num(r"solve_toeplitz の最大の差 ([\d.e+-]+)") < 1e-8, "[2] 自前と solve_toeplitz の差が 1e-8 未満")
snr0 = {k: num(rf"SNR\s+([-\d.]+) dB → wn_{k}\.wav") for k in ("noisy", "fir", "ml", "dd", "ideal_binary", "ideal_wiener")}
v = list(snr0.values())
ok(all(a < b for a, b in zip(v, v[1:])), "[3] 処理なし ＜ FIR ＜ ML ＜ DD ＜ 理想バイナリ ＜ 理想ウィーナー の順 " + str(snr0))
ok(abs(snr0["dd"] - 5.9) < 0.5 and abs(snr0["ideal_wiener"] - 9.7) < 0.5, "[3] DD 約 5.9 dB、理想ウィーナー 約 9.7 dB")

聞き比べます（手順 F の 13）。どれも同じ倍率で書き出してあるので、大きさの違いも聞こえます。`wn_ml.wav` に、ピコピコ・チリチリというミュージカルノイズが聞こえるか、`wn_dd.wav` で減るかをノートに書いてください。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['wn_noisy.wav', 'wn_ml.wav', 'wn_dd.wav', 'wn_ideal_wiener.wav', 'wn_clean.wav']:
    print(f)
    listen_file(f)

### SNR を変える（手順 F の 14）

`--snr 10` と `--snr -5` で実行し、理想ウィーナーマスクと、良いほうの推定マスク（ML と DD）の差を比べます。wav は上書きされます（それぞれ十数秒）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['wiener.py', '--snr', '10']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-4：ウィーナーフィルタ。時間領域の FIR 版（ウィーナー–ホップ方程式）と、STFT 上のマスク版を作り、聞き比べる

使い方: cd ~/home-lab/partIV/iv4 && uv run python wiener.py [音声の wav] [--snr 0]
  wav を指定しないときは librosa の例の音源 libri1（ダウンロードが必要）
やること
  0. 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1) のとき、E[s|x] = 0.8x、誤差の分散 0.8 を乱数で確かめる
  1. 音声の前に 1 秒の「雑音だけ」の区間をつけ、定常な雑音（色つき）を SNR 0 dB で足す
  2. 時間領域の FIR ウィーナーフィルタ（32 タップ）：ウィーナー–ホップ方程式 R h = r を
     自前（テプリッツ行列を作って np.linalg.solve）と scipy.linalg.solve_toeplitz で解いて照合。
     r = R_x − R_n（音声と雑音は無相関）なので、雑音だけの区間から R_n を推定すれば、正解の音声なしで作れる
  3. STFT のマスク：理想ウィーナーマスク（正解を使う。上限）、理想バイナリマスク、
     推定マスク（雑音の分散は最初の 1 秒、音声の分散は ML 推定 max(|x|² − λ_n, 0)、または decision-directed）
  4. 出力の SNR を比べ、wav に書き出す（wn_*.wav、どれも同じ倍率で。最初は小さな音量で聞く）。図 wiener.png
"""
import argparse
import pathlib
import sys

import numpy as np
import librosa
from scipy import linalg, signal
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft, istft                                      # IV-2 の共通ライブラリ
from common.audio_io import save                                         # 同上（1 を超える値は止める）

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(5)
FS, L, S = 16000, 512, 128
LEAD = FS                                          # 最初の 1 秒は雑音だけ
EPS = 1e-12


def snr_db(ref, est):
    """出力の SNR = 10 log10(Σ s² / Σ (s − ŝ)²)（尺度もそろっていることを求める、きびしい指標）"""
    return 10 * np.log10(np.sum(ref ** 2) / np.sum((ref - est) ** 2))


def autocorr(x, p):
    """標本自己相関 r̂[k] = (1/N) Σ x[n] x[n−k]、k = 0…p−1"""
    N = len(x)
    return np.array([np.dot(x[k:], x[:N - k]) / N for k in range(p)])


def fir_wiener(x, r_sx, p):
    """ウィーナー–ホップ方程式 Σ_k h[k] R_x[l−k] = r_sx[l]（l = 0…p−1）を自前で解く"""
    r = autocorr(x, p)
    R = np.array([[r[abs(i - j)] for j in range(p)] for i in range(p)])   # テプリッツ行列
    return np.linalg.solve(R, r_sx), r


def dd_mask(X, lam_n, alpha=0.98):
    """decision-directed 法（Ephraim–Malah）で事前 SNR ξ を推定し、ウィーナーゲイン ξ/(1+ξ) を返す"""
    F, T = X.shape
    G = np.zeros((F, T))
    prev = np.zeros(F)                                  # 1 つ前のフレームの |ŝ|²
    for t in range(T):
        gamma = np.abs(X[:, t]) ** 2 / lam_n            # 事後 SNR
        xi = alpha * prev / lam_n + (1 - alpha) * np.maximum(gamma - 1, 0)
        G[:, t] = xi / (1 + xi)
        prev = (G[:, t] * np.abs(X[:, t])) ** 2
    return G


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("wav", nargs="?")
    ap.add_argument("--snr", type=float, default=0.0)
    a = ap.parse_args()

    print("[0] 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1)、x = s + n")
    n0 = 200_000
    cn = lambda v, n: np.sqrt(v / 2) * (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
    s0, x0 = cn(4.0, n0), None
    x0 = s0 + cn(1.0, n0)
    g = np.sum(s0 * x0.conj()) / np.sum(np.abs(x0) ** 2)
    print(f"  最小二乗の係数 {g.real:.4f}{g.imag:+.4f}j（理論 λ_s/(λ_s+λ_n) = 0.8）、"
          f"誤差の分散 {np.mean(np.abs(s0 - 0.8 * x0) ** 2):.4f}（理論 λ_sλ_n/(λ_s+λ_n) = 0.8）")

    path = a.wav or librosa.ex("libri1")
    sp, _ = librosa.load(path, sr=FS, mono=True)
    s = np.r_[np.zeros(LEAD), sp / np.sqrt(np.mean(sp ** 2))]
    noise = signal.lfilter([1.0], [1.0, -0.7], RNG.standard_normal(len(s)))   # 定常な色つき雑音（AR(1)）
    act = s[LEAD:]
    noise *= np.sqrt(np.mean(act ** 2) / np.mean(noise ** 2)) / 10 ** (a.snr / 20)
    x = s + noise
    print(f"\n[1] {len(sp) / FS:.1f} 秒の音声＋1 秒の雑音だけの区間、入力の SNR {snr_db(s, x):.2f} dB（音声の区間で {a.snr} dB）")

    print("\n[2] 時間領域の FIR ウィーナーフィルタ（p = 32）")
    p = 32
    r_n = autocorr(noise[:LEAD], p)                    # 雑音だけの区間から R_n を推定
    r_x = autocorr(x, p)
    h_own, _ = fir_wiener(x, r_x - r_n, p)             # r_sx = R_x − R_n（無相関なので）
    h_sp = linalg.solve_toeplitz(r_x, r_x - r_n)
    print(f"  自前（np.linalg.solve）と scipy.linalg.solve_toeplitz の最大の差 {np.max(np.abs(h_own - h_sp)):.1e}（許容 1e-8）")
    assert np.max(np.abs(h_own - h_sp)) < 1e-8
    r_sx_oracle = np.array([np.dot(s[k:], x[:len(x) - k]) / len(x) for k in range(p)])   # 正解を使う版
    h_or, _ = fir_wiener(x, r_sx_oracle, p)
    y_fir = signal.lfilter(h_own, [1.0], x)
    y_or = signal.lfilter(h_or, [1.0], x)
    print(f"  出力の SNR：R_n を推定した版 {snr_db(s, y_fir):.2f} dB、正解の相互相関を使った版 {snr_db(s, y_or):.2f} dB")

    print("\n[3] STFT のマスク（L = 512、S = 128）")
    X, Sx, Nx = stft(x, L, S), stft(s, L, S), stft(noise, L, S)
    lam_n = np.mean(np.abs(X[:, : LEAD // S - L // S]) ** 2, axis=1)        # 最初の 1 秒のフレーム（音声と重ならない分）
    masks = {
        "ideal_wiener": np.abs(Sx) ** 2 / (np.abs(Sx) ** 2 + np.abs(Nx) ** 2 + EPS),
        "ideal_binary": (np.abs(Sx) > np.abs(Nx)).astype(float),
    }
    lam_s_ml = np.maximum(np.abs(X) ** 2 - lam_n[:, None], 0)                # 1 つの係数からの ML 推定
    masks["ml"] = lam_s_ml / (lam_s_ml + lam_n[:, None])
    masks["dd"] = dd_mask(X, lam_n[:, None][:, 0])
    out = {"noisy": x}
    for k, Mk in masks.items():
        out[k] = istft(Mk * X, L, S, "hann", length=len(x))
    out["fir"] = y_fir
    labels = {"noisy": "雑音入り（処理なし）", "fir": "FIR ウィーナー（R_n を推定）",
              "ideal_wiener": "理想ウィーナーマスク（正解を使う）", "ideal_binary": "理想バイナリマスク（正解を使う）",
              "ml": "推定マスク（ML、1 係数）", "dd": "推定マスク（decision-directed）"}
    scale = 0.03 / np.sqrt(np.mean(x ** 2))             # すべて同じ倍率で書き出す（大きさの違いも聞こえるように）
    res = {}
    for k, yk in out.items():
        res[k] = snr_db(s, yk)
        save(f"wn_{k}.wav", np.clip(yk * scale, -0.99, 0.99), FS)
        print(f"  {labels[k]:28s} SNR {res[k]:6.2f} dB → wn_{k}.wav")
    save("wn_clean.wav", np.clip(s * scale, -0.99, 0.99), FS)
    print("  （正解の音声は wn_clean.wav）")

    fig, ax = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
    ext = [0, X.shape[1] * S / FS, 0, FS / 2]
    db = lambda Z: 10 * np.log10(np.abs(Z) ** 2 + 1e-10)
    vmax = db(X).max()
    for a_, Z, t in ((ax[0, 0], X, "雑音入り |x|²（dB）"), (ax[0, 1], masks["dd"] * X, "推定マスク（DD）の出力（dB）")):
        a_.imshow(db(Z), origin="lower", aspect="auto", extent=ext, vmin=vmax - 70, vmax=vmax, cmap="magma")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    for a_, Mk, t in ((ax[1, 0], masks["ideal_wiener"], "理想ウィーナーマスク"), (ax[1, 1], masks["dd"], "推定マスク（DD）")):
        im = a_.imshow(Mk, origin="lower", aspect="auto", extent=ext, vmin=0, vmax=1, cmap="gray")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    fig.colorbar(im, ax=ax[1, :], shrink=0.8, label="ゲイン")
    fig.savefig("wiener.png", dpi=110)
    plt.show()   # ノートブック版：図を表示する
    print("\nwiener.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
snr10 = {k: num(rf"SNR\s+([-\d.]+) dB → wn_{k}\.wav") for k in ("ml", "dd", "ideal_wiener")}
gap10 = snr10["ideal_wiener"] - max(snr10["ml"], snr10["dd"])
print(f"+10 dB：理想ウィーナー − 良いほうの推定マスク = {gap10:.2f} dB")
ok(snr10["ml"] > snr10["dd"], "+10 dB では ML のほうが DD より良い")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['wiener.py', '--snr', '-5']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-4：ウィーナーフィルタ。時間領域の FIR 版（ウィーナー–ホップ方程式）と、STFT 上のマスク版を作り、聞き比べる

使い方: cd ~/home-lab/partIV/iv4 && uv run python wiener.py [音声の wav] [--snr 0]
  wav を指定しないときは librosa の例の音源 libri1（ダウンロードが必要）
やること
  0. 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1) のとき、E[s|x] = 0.8x、誤差の分散 0.8 を乱数で確かめる
  1. 音声の前に 1 秒の「雑音だけ」の区間をつけ、定常な雑音（色つき）を SNR 0 dB で足す
  2. 時間領域の FIR ウィーナーフィルタ（32 タップ）：ウィーナー–ホップ方程式 R h = r を
     自前（テプリッツ行列を作って np.linalg.solve）と scipy.linalg.solve_toeplitz で解いて照合。
     r = R_x − R_n（音声と雑音は無相関）なので、雑音だけの区間から R_n を推定すれば、正解の音声なしで作れる
  3. STFT のマスク：理想ウィーナーマスク（正解を使う。上限）、理想バイナリマスク、
     推定マスク（雑音の分散は最初の 1 秒、音声の分散は ML 推定 max(|x|² − λ_n, 0)、または decision-directed）
  4. 出力の SNR を比べ、wav に書き出す（wn_*.wav、どれも同じ倍率で。最初は小さな音量で聞く）。図 wiener.png
"""
import argparse
import pathlib
import sys

import numpy as np
import librosa
from scipy import linalg, signal
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt

sys.path.insert(0, ".")   # ノートブック版：common/ はこのノートブックの隣
from common.stft import stft, istft                                      # IV-2 の共通ライブラリ
from common.audio_io import save                                         # 同上（1 を超える値は止める）

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
RNG = np.random.default_rng(5)
FS, L, S = 16000, 512, 128
LEAD = FS                                          # 最初の 1 秒は雑音だけ
EPS = 1e-12


def snr_db(ref, est):
    """出力の SNR = 10 log10(Σ s² / Σ (s − ŝ)²)（尺度もそろっていることを求める、きびしい指標）"""
    return 10 * np.log10(np.sum(ref ** 2) / np.sum((ref - est) ** 2))


def autocorr(x, p):
    """標本自己相関 r̂[k] = (1/N) Σ x[n] x[n−k]、k = 0…p−1"""
    N = len(x)
    return np.array([np.dot(x[k:], x[:N - k]) / N for k in range(p)])


def fir_wiener(x, r_sx, p):
    """ウィーナー–ホップ方程式 Σ_k h[k] R_x[l−k] = r_sx[l]（l = 0…p−1）を自前で解く"""
    r = autocorr(x, p)
    R = np.array([[r[abs(i - j)] for j in range(p)] for i in range(p)])   # テプリッツ行列
    return np.linalg.solve(R, r_sx), r


def dd_mask(X, lam_n, alpha=0.98):
    """decision-directed 法（Ephraim–Malah）で事前 SNR ξ を推定し、ウィーナーゲイン ξ/(1+ξ) を返す"""
    F, T = X.shape
    G = np.zeros((F, T))
    prev = np.zeros(F)                                  # 1 つ前のフレームの |ŝ|²
    for t in range(T):
        gamma = np.abs(X[:, t]) ** 2 / lam_n            # 事後 SNR
        xi = alpha * prev / lam_n + (1 - alpha) * np.maximum(gamma - 1, 0)
        G[:, t] = xi / (1 + xi)
        prev = (G[:, t] * np.abs(X[:, t])) ** 2
    return G


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("wav", nargs="?")
    ap.add_argument("--snr", type=float, default=0.0)
    a = ap.parse_args()

    print("[0] 数値例：s ~ N_C(0, 4)、n ~ N_C(0, 1)、x = s + n")
    n0 = 200_000
    cn = lambda v, n: np.sqrt(v / 2) * (RNG.standard_normal(n) + 1j * RNG.standard_normal(n))
    s0, x0 = cn(4.0, n0), None
    x0 = s0 + cn(1.0, n0)
    g = np.sum(s0 * x0.conj()) / np.sum(np.abs(x0) ** 2)
    print(f"  最小二乗の係数 {g.real:.4f}{g.imag:+.4f}j（理論 λ_s/(λ_s+λ_n) = 0.8）、"
          f"誤差の分散 {np.mean(np.abs(s0 - 0.8 * x0) ** 2):.4f}（理論 λ_sλ_n/(λ_s+λ_n) = 0.8）")

    path = a.wav or librosa.ex("libri1")
    sp, _ = librosa.load(path, sr=FS, mono=True)
    s = np.r_[np.zeros(LEAD), sp / np.sqrt(np.mean(sp ** 2))]
    noise = signal.lfilter([1.0], [1.0, -0.7], RNG.standard_normal(len(s)))   # 定常な色つき雑音（AR(1)）
    act = s[LEAD:]
    noise *= np.sqrt(np.mean(act ** 2) / np.mean(noise ** 2)) / 10 ** (a.snr / 20)
    x = s + noise
    print(f"\n[1] {len(sp) / FS:.1f} 秒の音声＋1 秒の雑音だけの区間、入力の SNR {snr_db(s, x):.2f} dB（音声の区間で {a.snr} dB）")

    print("\n[2] 時間領域の FIR ウィーナーフィルタ（p = 32）")
    p = 32
    r_n = autocorr(noise[:LEAD], p)                    # 雑音だけの区間から R_n を推定
    r_x = autocorr(x, p)
    h_own, _ = fir_wiener(x, r_x - r_n, p)             # r_sx = R_x − R_n（無相関なので）
    h_sp = linalg.solve_toeplitz(r_x, r_x - r_n)
    print(f"  自前（np.linalg.solve）と scipy.linalg.solve_toeplitz の最大の差 {np.max(np.abs(h_own - h_sp)):.1e}（許容 1e-8）")
    assert np.max(np.abs(h_own - h_sp)) < 1e-8
    r_sx_oracle = np.array([np.dot(s[k:], x[:len(x) - k]) / len(x) for k in range(p)])   # 正解を使う版
    h_or, _ = fir_wiener(x, r_sx_oracle, p)
    y_fir = signal.lfilter(h_own, [1.0], x)
    y_or = signal.lfilter(h_or, [1.0], x)
    print(f"  出力の SNR：R_n を推定した版 {snr_db(s, y_fir):.2f} dB、正解の相互相関を使った版 {snr_db(s, y_or):.2f} dB")

    print("\n[3] STFT のマスク（L = 512、S = 128）")
    X, Sx, Nx = stft(x, L, S), stft(s, L, S), stft(noise, L, S)
    lam_n = np.mean(np.abs(X[:, : LEAD // S - L // S]) ** 2, axis=1)        # 最初の 1 秒のフレーム（音声と重ならない分）
    masks = {
        "ideal_wiener": np.abs(Sx) ** 2 / (np.abs(Sx) ** 2 + np.abs(Nx) ** 2 + EPS),
        "ideal_binary": (np.abs(Sx) > np.abs(Nx)).astype(float),
    }
    lam_s_ml = np.maximum(np.abs(X) ** 2 - lam_n[:, None], 0)                # 1 つの係数からの ML 推定
    masks["ml"] = lam_s_ml / (lam_s_ml + lam_n[:, None])
    masks["dd"] = dd_mask(X, lam_n[:, None][:, 0])
    out = {"noisy": x}
    for k, Mk in masks.items():
        out[k] = istft(Mk * X, L, S, "hann", length=len(x))
    out["fir"] = y_fir
    labels = {"noisy": "雑音入り（処理なし）", "fir": "FIR ウィーナー（R_n を推定）",
              "ideal_wiener": "理想ウィーナーマスク（正解を使う）", "ideal_binary": "理想バイナリマスク（正解を使う）",
              "ml": "推定マスク（ML、1 係数）", "dd": "推定マスク（decision-directed）"}
    scale = 0.03 / np.sqrt(np.mean(x ** 2))             # すべて同じ倍率で書き出す（大きさの違いも聞こえるように）
    res = {}
    for k, yk in out.items():
        res[k] = snr_db(s, yk)
        save(f"wn_{k}.wav", np.clip(yk * scale, -0.99, 0.99), FS)
        print(f"  {labels[k]:28s} SNR {res[k]:6.2f} dB → wn_{k}.wav")
    save("wn_clean.wav", np.clip(s * scale, -0.99, 0.99), FS)
    print("  （正解の音声は wn_clean.wav）")

    fig, ax = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
    ext = [0, X.shape[1] * S / FS, 0, FS / 2]
    db = lambda Z: 10 * np.log10(np.abs(Z) ** 2 + 1e-10)
    vmax = db(X).max()
    for a_, Z, t in ((ax[0, 0], X, "雑音入り |x|²（dB）"), (ax[0, 1], masks["dd"] * X, "推定マスク（DD）の出力（dB）")):
        a_.imshow(db(Z), origin="lower", aspect="auto", extent=ext, vmin=vmax - 70, vmax=vmax, cmap="magma")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    for a_, Mk, t in ((ax[1, 0], masks["ideal_wiener"], "理想ウィーナーマスク"), (ax[1, 1], masks["dd"], "推定マスク（DD）")):
        im = a_.imshow(Mk, origin="lower", aspect="auto", extent=ext, vmin=0, vmax=1, cmap="gray")
        a_.set(title=t, xlabel="時間 [s]", ylabel="周波数 [Hz]")
    fig.colorbar(im, ax=ax[1, :], shrink=0.8, label="ゲイン")
    fig.savefig("wiener.png", dpi=110)
    plt.show()   # ノートブック版：図を表示する
    print("\nwiener.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
snrm5 = {k: num(rf"SNR\s+([-\d.]+) dB → wn_{k}\.wav") for k in ("ml", "dd", "ideal_wiener")}
gapm5 = snrm5["ideal_wiener"] - max(snrm5["ml"], snrm5["dd"])
gap0 = snr0["ideal_wiener"] - max(snr0["ml"], snr0["dd"])
print("入力の SNR | 理想ウィーナー − 良いほうの推定マスク")
for s_, g_ in ((10, gap10), (0, gap0), (-5, gapm5)):
    print(f"  {s_:+3d} dB  | {g_:.2f} dB")
ok(abs(gap10 - 2.9) < 0.5 and abs(gap0 - 3.8) < 0.5 and abs(gapm5 - 4.2) < 0.5, "差は +10 dB で約 2.9、0 dB で約 3.8、−5 dB で約 4.2 dB")
ok(gap10 < gap0 < gapm5, "入力の SNR が低いほど、理想マスクと推定マスクの差が大きい")

## 7. 確認問題 10 のコード（IV-4-7）

確認問題（教科書の 7 節）は、**先に紙で解いてから**実行してください。問 10 の解答のコードです。教科書では `psd.py` の隣で `from psd import welch, FS` としますが、ここでは 2 節で定義した `welch` と `FS` をそのまま使います。

In [ ]:
import numpy as np
# from psd import welch, FS   # ノートブック版：2 節のセルで定義した welch・FS を使う
rng = np.random.default_rng(0)
pk = [10 * np.log10(np.max(welch(rng.standard_normal(16384), FS, 512)[1][1:-1] / (2 / FS))) for _ in range(300)]
print(f"最大のずれの平均 {np.mean(pk):+.2f} dB")
ok(abs(np.mean(pk) - 1.46) < 0.2, "問10 ウェルチ法の最大のずれの平均 約 +1.5 dB（教科書 1.46 dB）")

## おわりに

- 複素正規分布、自己相関とパワースペクトル密度、EM と補助関数法、音声の統計、ウィーナーフィルタを、自前の実装と既存のライブラリの照合、数値、音で確かめました。
- このノートブックに入れていないもの：自分の声（第7章の録音）での `speech_stats.py`・`wiener.py`（手順 E の 11）と、論文を読む手順 G。Mac のターミナルで、教科書の手順どおりに行ってください。